<a href="https://colab.research.google.com/github/rahmania15/Komputasi-Numerik/blob/main/SPL_GAUSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Penyelesaian SPL 3 Variabel dengan Eliminasi Gauss

**Sistem persamaan yang diselesaikan:**

| No | Persamaan |
|----|-----------|
| 1 | x + y + z = 4 |
| 2 | 2x − y + z = 0 |
| 3 | x + 2y − z = 9 |

## Cell 1: Memasukkan data soal

- `A` = matriks koefisien (setiap baris = satu persamaan; kolom = koefisien x, y, z).
- `b` = konstanta di ruas kanan tanda "=".
- Variabel tanpa angka (misalnya `x`) berarti koefisiennya **1**; tanda minus ditulis sebagai angka negatif.

In [ ]:
# Matriks koefisien A (3 x 3)
A = [[1,  1,  1],    # persamaan 1:  x +  y + z
     [2, -1,  1],    # persamaan 2: 2x -  y + z
     [1,  2, -1]]    # persamaan 3:  x + 2y - z

# Vektor konstanta b (ruas kanan)
b = [4, 0, 9]

print("Matriks koefisien A:")
for baris in A:
    print(baris)

print("\nVektor konstanta b:", b)

## Cell 2: Membuat matriks augmented

Matriks augmented `[A | b]` menggabungkan koefisien dan konstanta dalam satu tabel, sehingga setiap operasi baris otomatis berlaku untuk persamaan lengkapnya.

- `A[i][:]` menyalin baris supaya data asli `A` tidak berubah.
- `[b[i]]` ditempelkan sebagai kolom ke-4.
- Fungsi `tampilkan()` dibuat agar matriks bisa dicetak rapi berulang kali tanpa mengulang kode.

In [ ]:
n = len(b)   # jumlah persamaan = jumlah variabel = 3

# Gabungkan A dan b menjadi matriks augmented M
M = [A[i][:] + [b[i]] for i in range(n)]

def tampilkan(M, judul):
    """Mencetak matriks dengan rapi (3 angka desimal)."""
    print(judul)
    for baris in M:
        print("  ".join(f"{nilai:8.3f}" for nilai in baris))
    print()

tampilkan(M, "Matriks augmented awal [A | b]:")

## Cell 3: Eliminasi maju (membentuk segitiga atas)

Tujuan: membuat semua angka **di bawah diagonal** menjadi 0.

Untuk setiap kolom pivot `k`:
1. **Pivoting**: cari baris dengan nilai mutlak terbesar pada kolom `k`, lalu tukar ke posisi `k`. Ini mencegah pembagian dengan angka kecil/nol sehingga hasil lebih akurat.
2. Hitung **faktor** = elemen yang ingin dinolkan ÷ pivot.
3. **Operasi baris**: `baris_i = baris_i − faktor × baris_pivot`.

Contoh langkah 1: pivot terbesar di kolom 1 adalah `2` (baris 2), jadi baris 1 dan 2 ditukar. Lalu baris 2 baru = baris 2 − 0,5 × baris pivot, dan seterusnya.

In [ ]:
for k in range(n - 1):          # k = kolom pivot saat ini (0 lalu 1)

    # 1. PARTIAL PIVOTING: cari baris dengan |nilai| terbesar di kolom k
    p = max(range(k, n), key=lambda i: abs(M[i][k]))

    # 2. Jika pivot hampir nol, sistem tidak punya solusi tunggal
    if abs(M[p][k]) < 1e-12:
        raise ValueError("Pivot nol: sistem tidak punya solusi tunggal")

    # 3. Tukar baris k dengan baris pivot (jika perlu)
    if p != k:
        M[k], M[p] = M[p], M[k]
        tampilkan(M, f"Langkah {k+1}a: tukar baris {k+1} <-> baris {p+1}")

    # 4. Nolkan semua elemen di bawah pivot
    for i in range(k + 1, n):
        faktor = M[i][k] / M[k][k]            # pengali
        for j in range(k, n + 1):
            M[i][j] = M[i][j] - faktor * M[k][j]

    tampilkan(M, f"Langkah {k+1}b: setelah eliminasi kolom {k+1}")

## Cell 4: Substitusi mundur

Matriks sudah berbentuk segitiga atas, jadi baris terakhir hanya berisi satu variabel. Hitung dari **bawah ke atas**:

- Baris 3: `1,4 z = −1,4` → **z = −1**
- Baris 2: `2,5 y − 1,5 z = 9` → **y = 3**
- Baris 1: `2x − y + z = 0` → **x = 2**

`range(n - 1, -1, -1)` artinya menghitung mundur dari indeks 2, 1, lalu 0.

In [ ]:
x = [0] * n    # tempat menyimpan hasil [x, y, z]

for i in range(n - 1, -1, -1):
    # jumlahkan suku variabel yang sudah diketahui
    jumlah = sum(M[i][j] * x[j] for j in range(i + 1, n))
    # variabel ke-i = (konstanta - jumlah) / koefisien diagonal
    x[i] = (M[i][n] - jumlah) / M[i][i]

print("HASIL AKHIR:")
print(f"x = {x[0]:.4f}")
print(f"y = {x[1]:.4f}")
print(f"z = {x[2]:.4f}")

## Cell 5: Verifikasi hasil

Cara paling meyakinkan: masukkan x, y, z ke **persamaan awal**. Jika ruas kiri sama dengan ruas kanan, jawaban benar.

In [ ]:
A_asli = [[1, 1, 1], [2, -1, 1], [1, 2, -1]]
b_asli = [4, 0, 9]

print("VERIFIKASI:")
for i in range(3):
    ruas_kiri = sum(A_asli[i][j] * x[j] for j in range(3))
    cocok = abs(ruas_kiri - b_asli[i]) < 1e-9
    status = "BENAR ✓" if cocok else "SALAH ✗"
    print(f"Persamaan {i+1}: {ruas_kiri:.4f} = {b_asli[i]}  -> {status}")

## Cell 6 (opsional): Pembanding dengan NumPy

`np.linalg.solve` adalah fungsi bawaan NumPy untuk menyelesaikan SPL. Jika hasilnya sama dengan kode kita, berarti implementasi sudah benar.

In [ ]:
import numpy as np

hasil_numpy = np.linalg.solve(np.array(A_asli, dtype=float),
                              np.array(b_asli, dtype=float))
print("Hasil NumPy  :", hasil_numpy)
print("Hasil kodeku :", x)

## Kesimpulan

| Variabel | Nilai |
|----------|-------|
| x | **2** |
| y | **3** |
| z | **−1** |

Eliminasi Gauss berhasil menyelesaikan SPL 3 variabel.